# Plot parcellated features on a brain surface

In [4]:
!pip install nilearn
!pip install nibabel
!pip install brainspace

In [15]:
import os
import tempfile
import requests
import numpy as np
import nibabel as nib

from nilearn import plotting, datasets
from brainspace.utils.parcellation import map_to_labels

In [16]:
def fetch_annot(atlas, surf='fsaverage5'):
    '''
    Script that loads the labels of an specific parcellation and generates a midwall mask
    '''

    # Construct the URL for the surface file on GitHub
    url = f'https://github.com/MICA-MNI/micapipe/raw/refs/heads/master/parcellations'
      
    def download_atlas(url_to_file):
      # Download the surface file from the URL
      response = requests.get(url_to_file)

      # Ensure the request was successful
      if response.status_code != 200:
        raise Exception(f"Failed to download the {atlas} atlas on {surf} (Status code: {response.status_code})\n{url_to_file}")

      # Step 2: Save the downloaded file in a temporary directory
      with tempfile.NamedTemporaryFile(delete=False) as temp_file:
        temp_file.write(response.content)
        temp_file_name = temp_file.name  # Get the temporary file name
      
      return(temp_file_name)
    
    if surf == 'fsaverage5':
        # Load LEFT annotation file in fsaverage5
        tmp_file  = download_atlas(f'{url}/lh.{atlas}_mics.annot')
        annot_lh_fs5= nib.freesurfer.read_annot(tmp_file)
        os.remove(tmp_file)

        # Unique number of labels of a given atlas
        Ndim = max(np.unique(annot_lh_fs5[0]))

        # Load RIGHT annotation file in fsaverage5
        tmp_file  = download_atlas(f'{url}/rh.{atlas}_mics.annot')
        print(tmp_file)
        annot_rh_fs5 = nib.freesurfer.read_annot(tmp_file)[0]+Ndim
        os.remove(tmp_file)

        # replace with 0 the medial wall of the right labels
        annot_rh_fs5 = np.where(annot_rh_fs5==Ndim, 0, annot_rh_fs5) 

        # fsaverage5 labels
        labels = np.concatenate((annot_lh_fs5[0], annot_rh_fs5), axis=0)
    
    else:
        # Read label for fsLR-32k
        tmp_file  = download_atlas(f'{url}/{atlas}_conte69.csv')
        labels = np.loadtxt(open(tmp_file), dtype=int)
        os.remove(tmp_file)

    # mask of the medial wall
    mask = labels != 0
    
    # Midwall labels of aparc-a2009s are lh=42 and rh=117
    if atlas == 'aparc-a2009s' and surf == 'fsaverage5':
        mask[(labels == 117) | (labels == 42)] = 0
    
    #print(f'{atlas}; midwall = {Ndim}, length = {str(labels.shape)}')
    
    return(labels, mask, Ndim)

In [ ]:
# Load annotation file
annot = 'glasser-360'
labels, mask, Ndim = fetch_annot(annot, surf='fsaverage5')

# Load the Glasser Look Up Table (LUT)
# glasser = datasets.fetch_atlas_harvard_oxford('cort-maxprob-thr50-1mm')

# This is the brain's surface mesh (you can use 'inflated' or 'white' too)
surf_mesh = datasets.fetch_surf_fsaverage('fsaverage5')

# For example, use 'inflated' surface to show the brain's cortical surface
left_surf_mesh = surf_mesh['infl_left']
right_surf_mesh = surf_mesh['infl_right']



/var/folders/yv/7rdtqnkn0mb58f5x_yprm2gw0000gn/T/tmp47e2yf_j
[_add_readme_to_default_data_locations] Added README.md to /Users/rcruces/nilearn_data
[get_dataset_dir] Dataset created in /Users/rcruces/nilearn_data/fsl
[fetch_single_file] Downloading data from https://www.nitrc.org/frs/download.php/9902/HarvardOxford.tgz ...
Downloaded 5398528 of 25716861 bytes (21.0%%,    3.8s remaining)
Downloaded 11173888 of 25716861 bytes (43.4%%,    2.6s remaining)
Downloaded 16793600 of 25716861 bytes (65.3%%,    1.6s remaining)
Downloaded 21348352 of 25716861 bytes (83.0%%,    0.8s remaining)
[fetch_single_file]  ...done. (5 seconds, 0 min)

[uncompress_file] Extracting data from /Users/rcruces/nilearn_data/fsl/931d00c9505363ec4fd4eb167824cf39/HarvardOxford.tgz...
[uncompress_file] .. done.



{'filename': '/Users/rcruces/nilearn_data/fsl/data/atlases/HarvardOxford/HarvardOxford-cort-maxprob-thr50-1mm.nii.gz',
 'maps': <nibabel.nifti1.Nifti1Image at 0x14fc595e0>,
 'labels': ['Background',
  'Frontal Pole',
  'Insular Cortex',
  'Superior Frontal Gyrus',
  'Middle Frontal Gyrus',
  'Inferior Frontal Gyrus, pars triangularis',
  'Inferior Frontal Gyrus, pars opercularis',
  'Precentral Gyrus',
  'Temporal Pole',
  'Superior Temporal Gyrus, anterior division',
  'Superior Temporal Gyrus, posterior division',
  'Middle Temporal Gyrus, anterior division',
  'Middle Temporal Gyrus, posterior division',
  'Middle Temporal Gyrus, temporooccipital part',
  'Inferior Temporal Gyrus, anterior division',
  'Inferior Temporal Gyrus, posterior division',
  'Inferior Temporal Gyrus, temporooccipital part',
  'Postcentral Gyrus',
  'Superior Parietal Lobule',
  'Supramarginal Gyrus, anterior division',
  'Supramarginal Gyrus, posterior division',
  'Angular Gyrus',
  'Lateral Occipital Cort

In [ ]:
# Get the Binary degree centrality into a numpy array
matrix_centrality = np.sum(adj_matrix, axis=1, dtype=float)

# Map gradients to original parcels
feat_fsaverage5 = map_to_labels(matrix_centrality, labels, mask=mask, fill=np.nan)

# Visualizing the matrix on the left hemisphere
plotting.plot_surf_stat_map(left_surf_mesh, stat_map=feat_fsaverage5[0:10242], hemi='left', title="Left Degree centrality", cmap='rocket', vmax=78)

# Visualizing the matrix on the right hemisphere
plotting.plot_surf_stat_map(right_surf_mesh, stat_map=feat_fsaverage5[10242:], hemi='right', title="Right Degree centrality", cmap='rocket', vmax=78)

# Show the plots
plotting.show()
